# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivanjameslo/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

---
**My lane:** Refresh/Content Opportunity Scoring

**My primary ML task is ranking/scoring:** I want to prioritize content pages for human review using observable signals such as search impressions, CTR, search position, and content freshness.

A classification model could support this task by estimating whether a page is declining. Its score could then help rank eligible pages, with higher-priority pages appearing earlier in the review queue.

The intended output is a ranked list that helps a content or SEO reviewer decide which page to investigate first. It does not automatically determine whether a page should be edited.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import subprocess
import pandas as pd

#Step 1: Find or download the starter repository
repo_dir = "/content/flyrank-ml-internship-starter"

if not os.path.isdir(repo_dir):
  subprocess.run(
      [
          "git",
          "clone",
          "--depth",
          "1",
          "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
          repo_dir,
      ],
      check=True,
  )

#Step 2: Locate the CSV dataset
csv_path = os.path.join(
    repo_dir,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

#Step 3: Load the CSV into a DataFrame
df = pd.read_csv(csv_path)

#Step 4: Display the first few rows of the DataFrame and check its size
print(df.head(5))
print("Dataset shape: ", df.shape)

             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   
2               LOW  0.00  keyword article  informational      3515.0   
3               LOW  0.00  keyword article     commercial         NaN   
4               LOW  0.00  keyword article  informational      2803.0   

   char_count  ... char_count_tier   ctr  avg_position  engagement_rate  \
0     20457.0  ...     15000-25000  0.76 

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

---
**Target/Proxy:** Observed content decline

For this task, I will use the `trend_direction` column to construct a binary decline proxy. Pages labelled `down` will receive a value of `1`, while all other recorded trend categories will receive a value of `0`.

This proxy can support a classification model that estimates decline-related scores for ranking content pages.

However, observed decline is not the same as refresh priority. A declining page may not necessarily benefit from editing, and a non-declining page may still present an opportunity.

The starter dataset provides a current snapshot of content performance. It does not establish a future-decline target or prove which content actions would cause improvement. Future prediction would require properly aligned historical features and subsequent outcomes.

I will also avoid using `trend_pct` or `trend_direction` as predictive features because the decline proxy is derived from these columns. Including them would introduce target leakage.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Step 1: Create our binary decline proxy
df["decline_proxy"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

# Step 2: Display some examples
print("Sample labels:")
display(
    df[["trend_direction", "decline_proxy"]].head(10)
)

#Step 3: Count the two target categories
print("\nTarget distribution: ")
print(
    df["decline_proxy"].value_counts().sort_index()
)

Sample labels:


,trend_direction,decline_proxy
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1
5,down,1
6,down,1
7,stable,0
8,down,1
9,down,1



Target distribution: 
decline_proxy
0    13738
1    16262
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

---
**Primary success metric:** ***Precision@50***

I will use Precision@50 to evaluate how effectively a proposed ranking identifies pages labelled as declining among its top 50 recommendations.

This metric is relevant because a content or SEO reviewer has limited time and needs a manageable list of pages to investigate first.

I will compare the ranking against a simple hand-written baseline to determine whether the proposed method improves the quality of the review queue.

Since the target is a decline proxy, Precision@50 measures how well the system identifies recorded decline, not whether a refresh is necessary or would cause improved performance.

For a credible evaluation, the ranking should be tested on held-out data rather than only the pages used to develop it.

---
**Initial baseline observation:** Ranking pages by 90-day impressions alone produced a Precision@50 of 0.420, with 21 of the top 50 pages labelled as declining. This demonstrates how Precision@50 can measure the concentration of declining-labelled pages in a review queue. The result is an illustrative in-sample baseline, not evidence of performance on unseen clients or proof that these pages require refreshing.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Step 1: Create a simple baseline ranking
baseline = df.sort_values(
    "impressions_90d",
    ascending=False
)

# Step 2: Select the first 50 pages
top_50 = baseline.head(50)

# Step 3: Count the declining pages in the top 50
declining_count = top_50["decline_proxy"].sum()

# Step 4: Calculate Precision@50
precision_at_50 = declining_count / len(top_50)

# Step 5: Display the result
print("Pages reviewed: ", len(top_50))
print("Declining pages in top 50: ", declining_count)
print(f"Baseline Precision@50: {precision_at_50:.3f}")

Pages reviewed:  50
Declining pages in top 50:  21
Baseline Precision@50: 0.420


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

---
***One row = one anonymizedd content page***

Each row in the starter dataset represent one content page, identified by a pseudonymized `content_id`. The columns decribe characteristics such as search impressions, CTR, average search positions, content age, and time since the last update.

For this task, these characteristics can serve as candidate features, while `decline_proxy` represents the observed decline label.

The dataset contains 30,000 content pages. The intended output is a priority score for each eligible page, which can be used to create a ranked review queue.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Step 1: Choose the columns relevant to our task
columns = [
    "content_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "decline_proxy"
]

# Step 2: Create a smaller DataFrame
page_data = df[columns].copy()

# Step 3: Display the first five pages
print("Unit of analysis: One anonymized content page")
print("Dataset shape: ", page_data.shape)

display(page_data.head(5))

# Step 4: Check whether each content ID is unique
print("Unique content IDs: ", page_data["content_id"].nunique())
print("Total rows: ", len(page_data))


Unit of analysis: One anonymized content page
Dataset shape:  (30000, 7)


,content_id,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update,decline_proxy
0,content_304f48230142,3803,0.76,10.6,187,20,1
1,content_a1fb4e703a9e,15320,0.05,20.3,445,25,1
2,content_9aa793d4d895,12581,0.09,36.5,141,20,1
3,content_331d6c4de07b,11751,0.49,6.2,463,22,0
4,content_d99b7a2d90ca,19140,0.13,44.0,263,14,1


Unique content IDs:  30000
Total rows:  30000


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

---
**Why ML may improve on a fixed rule**

A fixed rule, such as selecting apges that have not been updated for at least 180 days and have at least 500 impressions, is simple and interpretable. However, it only considers a limited combination of conditions.

A machine-learning approach could learn patterns involving multiple signals, including impressions, CTR, search position, content age, and update frequency. It could also assign different scores to pages, helping reviewers prioritize a larger pool of candidates.

I would compare the proposed approach against a fixed-rule baseline using Precision@50 on held-out data. ML would only be justified if it provides a meaningful improvement for the review decision.

The resulting scores would support human review rather than automatically determine which pages should be edited.

---
**Initial fixed-rule observation:** The stale-and-visible rule selected 17 pages from the 30,000-page dataset. Of these, 16 were labelled as declining, resulting in a decline rate of 94.1%.

Although the rule identified a small group with a high concentration of declining pages, it excluded most other pages labelled as declining. A scoring approach could help prioritize a broader set of review candidates using multiple signals.

These results do not establish that ML outperforms the fixed rule. A fair comparison would require a common evaluation set, a consistent ranking size, and held-out validation.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Step 1: Define our fixed rule
fixed_rule = (
    (df["days_since_last_update"] >= 180)
    & (df["impressions_90d"] >= 500)
)

# Step 2: Select pages that meet both conditions
rule_candidates = df[fixed_rule]

# Step 3: Count the qualifying pages
candidate_count = len(rule_candidates)

# Step 4: Count how many are labelled declining
declining_candidates = rule_candidates["decline_proxy"].sum()

# Step 5: Calculate the decline rate
if candidate_count > 0:
  decline_rate = declining_candidates / candidate_count
else:
  decline_rate = 0

# Step 6: Display the result
print("Pages selected by fixed rule: ", candidate_count)
print("Declining pages among candidates: ", declining_candidates)
print(f"Decline rate among candidates: {decline_rate:.3f}")

Pages selected by fixed rule:  17
Declining pages among candidates:  16
Decline rate among candidates: 0.941


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.